# 🧬 Single-Cell RNA-seq Analysis with Scanpy

This notebook walks through a **complete single-cell RNA-seq (scRNA-seq)** data analysis pipeline using the Python package **Scanpy**.  
It is written for **beginners** in single-cell bioinformatics — especially those coming from a biology background.

We'll go step-by-step through:
1. Loading and preprocessing data  
2. Quality control (QC)  
3. Normalization and feature selection  
4. Dimensionality reduction (PCA, UMAP)  
5. Clustering  
6. Marker gene detection and visualization

By the end of this notebook, you’ll understand not only *how* to perform each step, but also *why* it’s important.

---


### Installation

In [ ]:
!pip install scanpy

In [ ]:
!pip install anndata

In [ ]:
!pip3 install igraph

In [ ]:
!pip install celltypist

In [ ]:
!pip install decoupler

In [ ]:
!pip install fa2-modified

### 🧩 Loading Data
In this step, we load the single-cell expression matrix into an **AnnData** object — the core data structure in Scanpy.
It contains:
- `adata.X`: the expression matrix (cells × genes)
- `adata.obs`: metadata for each cell
- `adata.var`: metadata for each gene


In [ ]:
#Import core single cell tools

import scanpy as sc
import anndata as ad

In [ ]:
!wget https://cf.10xgenomics.com/samples/cell-vdj/8.0.1/10k_5p_Human_diseased_PBMC_ALL_Fresh/10k_5p_Human_diseased_PBMC_ALL_Fresh_count_filtered_feature_bc_matrix.h5

In [ ]:
fresh_blood_adata = sc.read_10x_h5('10k_5p_Human_diseased_PBMC_ALL_Fresh_count_filtered_feature_bc_matrix.h5')

In [ ]:
print(fresh_blood_adata)

In [ ]:
# the dimensions of our dataset
fresh_blood_adata.shape

In [ ]:
#13853 cells
#38606 genes

In [ ]:
# let's look at the first 5 rows describing the genes in our dataset
fresh_blood_adata.var.head()

In [ ]:
# let's look at the first 5 rows describing the cells (ID) in our dataset
fresh_blood_adata.obs.head()

In [ ]:
# How about both. in a proper dataframe format

fresh_blood_adata.to_df()

### 🧹 Quality Control (QC)
QC ensures we only keep high-quality cells and informative genes.
Typical filters remove:
- Harmonize unique gene names (avoid gene duplications from old pipelines)
- Cells with too few genes (likely dead)
- Cells with too many genes (possible doublets)
- Genes expressed in very few cells (uninformative)


In [ ]:
# A useful step for older datasets
fresh_blood_adata.var_names_make_unique()
fresh_blood_adata.obs_names_make_unique()

In [ ]:
#Let's search for possible contamination from dying cells, ribosomal transcripts or hemoglobin

#Cells with a high proportion of mitochondrial reads (say >10–20%) are likely stressed, apoptotic, or poorly captured
#Ribosomal transcripts are removed because they represent global transcriptional activity, not cell-type-specific biology
#Instead of true cell populations, high HB signal often represents ambient RNA contamination from lysed red blood cells

fresh_blood_adata.var['MT'] = fresh_blood_adata.var_names.str.startswith("MT-")
fresh_blood_adata.var['RIBO'] = fresh_blood_adata.var_names.str.startswith("RPS", "RPL")
fresh_blood_adata.var['HB'] = fresh_blood_adata.var_names.str.startswith("^HB[^(P)]")

In [ ]:
#let's just take a quick look at one of them.

mt_genes = fresh_blood_adata.var[fresh_blood_adata.var['MT']]
mt_genes

In [ ]:
#calculate the qc metrics

sc.pp.calculate_qc_metrics(
    fresh_blood_adata, qc_vars=["MT", 'RIBO', 'HB'], inplace=True, log1p=True
)

In [ ]:
#note that it is also included in the headers of obs

fresh_blood_adata.obs.head()

In [ ]:
#and your gene list
fresh_blood_adata.var.head()


In [ ]:
#what is the average number of genes that have at least one detected identifier in each cell.
#in other words, the number of genes expressed in each cell

sc.pl.violin(
    fresh_blood_adata,
    ["n_genes_by_counts"],
    jitter=0.4,
    multi_panel=False,
)

In [ ]:
#What is the total number of molecules (UMI) detected in a cell.
#basically you can have 10 UMI molecules but they are all pointing to the same gene.

sc.pl.violin(
    fresh_blood_adata,
    ["total_counts"],
    jitter=0.4,
    multi_panel=False,
)

In [ ]:
#what about those mitochondrial genes?
sc.pl.violin(
    fresh_blood_adata,
    ["pct_counts_MT"],
    jitter=0.4,
    multi_panel=False,
)

In [ ]:
#and the ribosomal genes
sc.pl.violin(
    fresh_blood_adata,
    ["pct_counts_RIBO"],
    jitter=0.4,
    multi_panel=False,
)

In [ ]:
#let's visualize the three of them. And let's see where the mitochondrial genes are
sc.pl.scatter(fresh_blood_adata, "total_counts", "n_genes_by_counts", color="pct_counts_MT")

In [ ]:
sc.pl.scatter(fresh_blood_adata, "total_counts", "n_genes_by_counts", color="pct_counts_RIBO")

In [ ]:
sc.pl.scatter(fresh_blood_adata, "total_counts", "n_genes_by_counts", color="pct_counts_HB")

In [ ]:
"""
Additionally, it is important to note that for datasets with multiple batches,
quality control should be performed for each sample individually as quality
control thresholds can vary substantially between batches.
"""

In [ ]:
#Let's use the data MT plot to select things to remove
sc.pp.filter_cells(fresh_blood_adata, min_genes=1000)
sc.pp.filter_genes(fresh_blood_adata, min_cells=1000)

In [ ]:
sc.pl.scatter(fresh_blood_adata, "total_counts", "n_genes_by_counts", color="pct_counts_MT")

In [ ]:
# we can also further filter for ribosomal contaminations using

fresh_blood_adata = fresh_blood_adata[
    fresh_blood_adata.obs['pct_counts_RIBO'] < 10,
    :
]


In [ ]:
sc.pl.scatter(fresh_blood_adata, "total_counts", "n_genes_by_counts", color="pct_counts_RIBO")

In [ ]:
#doublet detection
##Identifying doublets is crucial as they can lead to misclassifications or
##distortions in downstream analysis steps

In [ ]:
sc.pp.scrublet(fresh_blood_adata) #if you have multiple batch samples, you can specify it with batch_key="sample"

### ⚖️ Normalization
Normalization adjusts for sequencing depth differences between cells.
Here, we scale counts so each cell has the same total expression level.

In [ ]:
#Normalization

In [ ]:
# Save a copy of the data
fresh_blood_adata.layers["counts"] = fresh_blood_adata.X.copy()

In [ ]:
# Normalizing to median total counts
sc.pp.normalize_total(fresh_blood_adata)
# Logarithmize the data
sc.pp.log1p(fresh_blood_adata)

In [ ]:
#Feature selection
#selecting the top 1000 most variable genes
sc.pp.highly_variable_genes(fresh_blood_adata, n_top_genes=1000)

In [ ]:
sc.pl.highly_variable_genes(fresh_blood_adata )
#left is normalized
#right is not

### 🔍 Dimensionality Reduction (PCA)
We use **Principal Component Analysis (PCA)** to reduce data complexity and highlight key variation patterns.
This makes later steps like clustering and visualization faster and more robust.

In single-cell RNA-seq, each cell has expression values for thousands of genes, creating a huge, noisy matrix. PCA compresses this high-dimensional data into a smaller set of features (typically 30–50 components) that summarize the key biological and technical variation across cells.

- Noise reduction: scRNA-seq data are sparse and noisy. PCA focuses on the strongest correlated gene expression patterns, discarding random noise.

- Computational efficiency: Downstream analyses like clustering, UMAP, or t-SNE run much faster and more robustly on 30 PCs than on 20,000 genes.

- Signal extraction: The top PCs often correspond to meaningful biological structure—cell type, cell cycle state, or activation level—while later PCs capture less relevant variation.

In [ ]:
sc.tl.pca(fresh_blood_adata)

In [ ]:
sc.pl.pca_variance_ratio(fresh_blood_adata, n_pcs=10, log=False)

In [ ]:
sc.pl.pca(
    fresh_blood_adata,
    color=["pct_counts_MT"]
)

In [ ]:
## Nearest Neighbour
# Let us compute the neighborhood graph of cells using the PCA representation of the data matrix.
# basically we want to cluster the PCA components

In [ ]:
sc.pp.neighbors(fresh_blood_adata)

In [ ]:
sc.tl.umap(fresh_blood_adata)

In [ ]:
sc.pl.umap(
    fresh_blood_adata,
    color=["pct_counts_RIBO"],
    size=8,
)

In [ ]:
## Clustering by communities.

##Clustering by communities in single-cell RNA-seq is the process of grouping cells that show similar expression profiles — essentially, discovering putative cell types or states.

## Once PCA compresses your data into a manageable set of dimensions, clustering algorithms like Leiden operate on a graph-based representation of cell–cell relationships.

## Usually used for cell type detection

In [ ]:
# Using the igraph implementation and a fixed number of iterations can be significantly faster, especially for larger datasets
sc.tl.leiden(fresh_blood_adata, flavor="igraph", n_iterations=2)

In [ ]:
sc.pl.umap(
    fresh_blood_adata,
    color=["pct_counts_RIBO"],
    size=8,
)

In [ ]:
sc.pl.umap(
    fresh_blood_adata,
    color=["leiden"],
    size=8,
)

In [ ]:
sc.pl.umap(
    fresh_blood_adata,
    color=["leiden"],
    # increase horizontal space between panels
    wspace=0.5,
    size=3,
    ncols = 1
)

In [ ]:
sc.pl.umap(
    fresh_blood_adata,
    color=[ "predicted_doublet"],
    # increase horizontal space between panels
    wspace=0.5,
    size=3,
    ncols = 1
)

In [ ]:
sc.pl.umap(
    fresh_blood_adata,
    color=[ "doublet_score"],
    # increase horizontal space between panels
    wspace=0.5,
    size=3,
    ncols = 1
)

In [ ]:
#Further reclustering

In [ ]:
sc.tl.leiden(fresh_blood_adata, flavor="igraph", n_iterations=2, key_added="leiden_res0_02", resolution=0.02)
sc.tl.leiden(fresh_blood_adata, flavor="igraph", n_iterations=2, key_added="leiden_res0_5", resolution=0.5)
sc.tl.leiden(fresh_blood_adata, flavor="igraph", n_iterations=2, key_added="leiden_res2", resolution=2)

In [ ]:
sc.pl.umap(
    fresh_blood_adata,
    color=["leiden_res0_02"],
    # increase horizontal space between panels
    wspace=0.5,
    size=15,
    ncols = 1
)

In [ ]:
sc.pl.umap(
    fresh_blood_adata,
    color=["leiden_res0_5"],
    # increase horizontal space between panels
    wspace=0.5,
    size=15,
    ncols = 1,
    legend_loc="on data"
)

In [ ]:
sc.pl.umap(
    fresh_blood_adata,
    color=["leiden_res2"],
    # increase horizontal space between panels
    wspace=0.5,
    size=15,
    ncols = 1,
    legend_loc="on data"
)

### Cell Annotation
Cell annotation is the process of assigning biological meaning—like cell type or functional state—to each cluster found after Leiden clustering.

Traditionally, this relies on manual marker gene inspection: you identify top genes per cluster and match them to known markers. But tools like Decoupler enable a more systematic and data-driven approach.


Decoupler is a framework for gene set activity inference. Instead of labeling clusters by single markers, it estimates the activity of predefined pathways, transcription factors, or cell-type signatures from known databases (e.g., MSigDB, PROGENy, DoRothEA).

In practice:

- You provide your normalized expression matrix (adata).

- You load gene sets representing known biological programs or cell-type signatures.

- Decoupler calculates an activity score per cell or cluster using methods like weighted mean, ULM, or AUCell.

- You interpret those activities to annotate clusters automatically or semi-automatically.

In [ ]:
import decoupler as dc

In [ ]:
# Query Omnipath and get PanglaoDB
markers = dc.op.resource(name="PanglaoDB", organism="human")

# Keep canonical cell type markers alone
markers = markers[markers["canonical_marker"]]

# Remove duplicated entries
markers = markers[~markers.duplicated(["cell_type", "genesymbol"])]

# Format because dc only accepts cell_type and genesymbol

markers = markers.rename(columns={"cell_type": "source", "genesymbol": "target"})
markers = markers[["source", "target"]]


markers.head()

In [ ]:
#load the gene expression matrix into dc

dc.mt.ulm(data=fresh_blood_adata,
          net=markers,
          tmin = 3)

In [ ]:
#retrieve the score for each cell type

score = dc.pp.get_obsm(fresh_blood_adata, key="score_ulm")
score

In [ ]:
#preview the data
fresh_blood_adata.obsm["score_ulm"].head()

In [ ]:
fresh_blood_adata.obsm["score_ulm"].columns

In [ ]:
sc.pl.umap(score, color=["B cells memory", "leiden_res0_02"], cmap="RdBu_r")

In [ ]:
import seaborn as sns

In [ ]:
sc.pl.violin(score, keys=["B cells memory"], groupby="leiden_res0_02", rotation=90)

In [ ]:
sc.pl.violin(score, keys=["B cells memory"], groupby="leiden_res0_02", rotation=90)

In [ ]:
#. Now let's know what each of the 7 clusters mean

In [ ]:
#rank genes
fresh_blood_adata_rank = dc.tl.rankby_group(score, groupby="leiden_res0_02", reference="rest", method="t-test_overestim_var")
fresh_blood_adata_rank = fresh_blood_adata_rank[fresh_blood_adata_rank["stat"] > 0]
fresh_blood_adata_rank.head()

In [ ]:
cluster_annotations = fresh_blood_adata_rank[fresh_blood_adata_rank["stat"] > 0].groupby("group").head(1).set_index("group")["name"].to_dict()

In [ ]:
cluster_annotations

In [ ]:
fresh_blood_adata.obs['cell_type'] = fresh_blood_adata.obs['leiden_res0_02'].map(cluster_annotations)


In [ ]:
# Example of how to subset for multiple genes in the 'source' column
available_genes = set(fresh_blood_adata.var_names)

b_cell_markers = markers[markers['source'].isin(['B cells memory'])]['target']
b_cell_markers = b_cell_markers[b_cell_markers.isin(available_genes)]

nk_cell_markers = markers[markers['source'].isin(['Natural killer T cells'])]['target']
nk_cell_markers = nk_cell_markers[nk_cell_markers.isin(available_genes)]

t_cells_markers = markers[markers['source'].isin(['T cells'])]['target']
t_cells_markers = t_cells_markers[t_cells_markers.isin(available_genes)]


#display(b_cell_markers)

### Other ways to visualize the cell types

In [ ]:
marker_genes_dict = {
    "B cells": b_cell_markers.head().tolist(),
    "NK cells": nk_cell_markers.head().tolist(),
    "T cells": t_cells_markers.head().tolist()
}

In [ ]:
sc.pl.dotplot(fresh_blood_adata, marker_genes_dict, "cell_type", dendrogram=True)

In [ ]:
sc.pl.stacked_violin(
    fresh_blood_adata, marker_genes_dict, groupby="leiden_res0_02",  dendrogram=True
)

In [ ]:
sc.pl.matrixplot(
    fresh_blood_adata,
    marker_genes_dict,
    "leiden_res0_02",
    dendrogram=True,
    cmap="Blues",
)

In [ ]:
sc.pl.heatmap(
    fresh_blood_adata, marker_genes_dict, groupby="leiden_res0_02", cmap="viridis", dendrogram=True
)

In [ ]:
# @title Using genome tracks
sc.pl.tracksplot(fresh_blood_adata, marker_genes_dict, groupby="leiden_res0_02", dendrogram=False)